# Baseline Benchmarks for Bayesian Role Specialization Model

Compares the Bayesian model against 7 baselines:
1. **Random** — uniform 1/27 over all role combos
2. **Random Walk** — stick to previous role (1-ε), switch randomly (ε); ε fitted to maximize combo_r
3. **Optimal** — always predict the value-optimal role combo
4. **Random Among Top-k** — uniform over top-k combos by value; k fitted to maximize combo_r
5. **Random-to-Optimal** — linear interpolation from uniform to optimal over stages
6. **Copy Others** — each agent plays the role others played last stage
7. **Contradict Others** — each agent plays a role others did NOT play last stage

All baselines are fitted to maximize **combo_r** (Pearson r on combo-level predictions).

In [1]:
import json
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*constant.*")
warnings.filterwarnings("ignore", message=".*nearly constant.*")

from online_model_sim import (
    ALL_ROLE_COMBOS,
    ROLE_CHAR_TO_IDX,
    ROLE_NAMES,
    canonical_combo,
    combo_marginal,
    get_canonical_combos,
    load_team_rounds,
    run_all_predictions,
    compute_pearson,
)

## Load Human Data

In [2]:
records = load_team_rounds(data_dirs=["../bayesian-role-specialization-2026-03-18-15-47-09"])
n_envs = len(set(r["env_id"] for r in records))
print(f"Loaded {len(records)} team-rounds across {n_envs} envs")


Metal device set to: Apple M4
Loaded 48 team-rounds across 6 envs


W0000 00:00:1774118688.291258 3386063 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1774118688.301422 3386063 service.cc:145] XLA service 0xafb6b6100 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774118688.301430 3386063 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1774118688.302382 3386063 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1774118688.302389 3386063 mps_client.cc:384] XLA backend will use up to 12712640512 bytes on device 0 for SimpleAllocator.


## Helper Functions

In [3]:
def _dist_to_marginal(dist):
    marginal = np.zeros(3)
    for combo_str, prob in dist.items():
        if prob > 0:
            marginal += prob * combo_marginal(combo_str)
    return marginal


def run_baseline_predictions(records, baseline_fn, **kwargs):
    """Run a baseline and aggregate like the main model."""
    by_env = defaultdict(list)
    for rec in records:
        by_env[rec["env_id"]].append(rec)

    all_results = {}
    for env_id, env_records in by_env.items():
        stat_profile = env_records[0]["stat_profile"]
        optimal = env_records[0]["optimal_roles"]
        canon_combos = get_canonical_combos(stat_profile)

        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        team_predictions = []
        max_stages = 0

        for rec in env_records:
            preds = baseline_fn(rec, **kwargs)
            team_predictions.append(preds)
            for s, pred in enumerate(preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred["predicted_dist"].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred["human_combo"], stat_profile)] += 1
                stage_model_marg[s] += pred["model_marginal"]
                stage_human_marg[s] += combo_marginal(pred["human_combo"])

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        all_results[env_id] = {
            "stat_profile": stat_profile, "optimal": optimal,
            "canonical_optimal": canonical_combo(optimal, stat_profile),
            "canonical_combos": canon_combos, "n_teams": len(env_records),
            "max_stages": max_stages, "stage_predicted": predicted_avg,
            "stage_human": dict(stage_human), "stage_counts": dict(stage_counts),
            "team_predictions": team_predictions,
            "stage_model_marginal": model_marg_avg,
            "stage_human_marginal": human_marg_avg,
        }
    return all_results


def extract_metrics(correlations):
    """Extract global and per-env combo_r and marg_r."""
    g = correlations.get("__global__", {})
    global_m = {
        "combo_r": g.get("combo", {}).get("r", float("nan")),
        "marg_r": g.get("marginal", {}).get("r", float("nan")),
    }
    per_env = {}
    for env_id, c in correlations.items():
        if env_id == "__global__":
            continue
        per_env[env_id] = {
            "combo_r": c.get("combo", {}).get("r", float("nan")),
            "marg_r": c.get("marginal", {}).get("r", float("nan")),
        }
    return global_m, per_env

## Baseline Definitions

In [4]:
# === 1. Random ===
def random_predictions(record):
    dist = {c: 1.0 / 27.0 for c in ALL_ROLE_COMBOS}
    marginal = np.array([1/3, 1/3, 1/3])
    return [{"predicted_dist": dict(dist), "human_combo": hc, "model_marginal": marginal.copy()}
            for hc in record["stage_roles"]]


# === 2. Random Walk ===
def random_walk_predictions(record, eps):
    results = []
    for s, human_combo in enumerate(record["stage_roles"]):
        if s == 0:
            per_agent = [np.array([1/3, 1/3, 1/3]) for _ in range(3)]
        else:
            prev_combo = record["stage_roles"][s - 1]
            per_agent = []
            for i in range(3):
                prev_role = ROLE_CHAR_TO_IDX[prev_combo[i]]
                probs = np.full(3, eps / 3.0)
                probs[prev_role] += 1.0 - eps
                per_agent.append(probs)
        dist = {}
        for combo_str in ALL_ROLE_COMBOS:
            r0, r1, r2 = [ROLE_CHAR_TO_IDX[c] for c in combo_str]
            dist[combo_str] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])
        results.append({"predicted_dist": dist, "human_combo": human_combo,
                        "model_marginal": np.mean(per_agent, axis=0)})
    return results


# === 3. Optimal ===
def _optimal_combo_dist(values, intent, team_hp, enemy_hp):
    vals = values[:, intent, team_hp, enemy_hp]
    max_val = vals.max()
    optimal_mask = vals == max_val
    n_optimal = int(optimal_mask.sum())
    dist = {}
    for combo_str in ALL_ROLE_COMBOS:
        flat_idx = ROLE_CHAR_TO_IDX[combo_str[0]] * 9 + ROLE_CHAR_TO_IDX[combo_str[1]] * 3 + ROLE_CHAR_TO_IDX[combo_str[2]]
        dist[combo_str] = 1.0 / n_optimal if optimal_mask[flat_idx] else 0.0
    return dist

def optimal_predictions(record):
    env = record["env_config"]
    values = env["values"]
    initial_intent = record["lds"][0] if record["lds"] else 0
    dist = _optimal_combo_dist(values, initial_intent, env["team_max_hp"], env["enemy_max_hp"])
    marginal = _dist_to_marginal(dist)
    return [{"predicted_dist": dict(dist), "human_combo": hc, "model_marginal": marginal.copy()}
            for hc in record["stage_roles"]]


# === 4. Random Among Top-k ===
def topk_predictions(record, k):
    env = record["env_config"]
    values = env["values"]
    initial_intent = record["lds"][0] if record["lds"] else 0
    vals = values[:, initial_intent, env["team_max_hp"], env["enemy_max_hp"]]
    top_indices = np.argsort(vals)[::-1][:k]
    dist = {}
    for combo_str in ALL_ROLE_COMBOS:
        flat_idx = ROLE_CHAR_TO_IDX[combo_str[0]] * 9 + ROLE_CHAR_TO_IDX[combo_str[1]] * 3 + ROLE_CHAR_TO_IDX[combo_str[2]]
        dist[combo_str] = 1.0 / k if flat_idx in top_indices else 0.0
    marginal = _dist_to_marginal(dist)
    return [{"predicted_dist": dict(dist), "human_combo": hc, "model_marginal": marginal.copy()}
            for hc in record["stage_roles"]]


# === 5. Random-to-Optimal ===
def random_to_optimal_predictions(record):
    env = record["env_config"]
    values = env["values"]
    initial_intent = record["lds"][0] if record["lds"] else 0
    optimal_dist = _optimal_combo_dist(values, initial_intent, env["team_max_hp"], env["enemy_max_hp"])
    uniform_dist = {c: 1.0 / 27.0 for c in ALL_ROLE_COMBOS}
    n_stages = len(record["stage_roles"])
    results = []
    for s, human_combo in enumerate(record["stage_roles"]):
        alpha = s / max(n_stages - 1, 1)
        interp_dist = {c: (1 - alpha) * uniform_dist[c] + alpha * optimal_dist[c] for c in ALL_ROLE_COMBOS}
        marginal = _dist_to_marginal(interp_dist)
        results.append({"predicted_dist": interp_dist, "human_combo": human_combo, "model_marginal": marginal})
    return results


# === 6. Copy Others ===
def copy_others_predictions(record):
    results = []
    for s, human_combo in enumerate(record["stage_roles"]):
        if s == 0:
            per_agent = [np.array([1/3, 1/3, 1/3]) for _ in range(3)]
        else:
            prev_combo = record["stage_roles"][s - 1]
            per_agent = []
            for i in range(3):
                others = [ROLE_CHAR_TO_IDX[prev_combo[j]] for j in range(3) if j != i]
                probs = np.zeros(3)
                for r in others:
                    probs[r] += 0.5
                per_agent.append(probs)
        dist = {}
        for combo_str in ALL_ROLE_COMBOS:
            r0, r1, r2 = [ROLE_CHAR_TO_IDX[c] for c in combo_str]
            dist[combo_str] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])
        results.append({"predicted_dist": dist, "human_combo": human_combo,
                        "model_marginal": np.mean(per_agent, axis=0)})
    return results


# === 7. Contradict Others ===
def contradict_others_predictions(record):
    results = []
    for s, human_combo in enumerate(record["stage_roles"]):
        if s == 0:
            per_agent = [np.array([1/3, 1/3, 1/3]) for _ in range(3)]
        else:
            prev_combo = record["stage_roles"][s - 1]
            per_agent = []
            for i in range(3):
                others_roles = set(ROLE_CHAR_TO_IDX[prev_combo[j]] for j in range(3) if j != i)
                different_roles = [r for r in range(3) if r not in others_roles]
                if not different_roles:
                    different_roles = [0, 1, 2]
                probs = np.zeros(3)
                for r in different_roles:
                    probs[r] = 1.0 / len(different_roles)
                per_agent.append(probs)
        dist = {}
        for combo_str in ALL_ROLE_COMBOS:
            r0, r1, r2 = [ROLE_CHAR_TO_IDX[c] for c in combo_str]
            dist[combo_str] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])
        results.append({"predicted_dist": dist, "human_combo": human_combo,
                        "model_marginal": np.mean(per_agent, axis=0)})
    return results

## Run Bayesian Model (Finetuned Params)

In [5]:
MAIN_TAU_PRIOR = 5.6
MAIN_TAU_SOFTMAX = 0.1
MAIN_EPSILON = 0.2

main_results = run_all_predictions(records, tau_prior=MAIN_TAU_PRIOR, tau_softmax=MAIN_TAU_SOFTMAX, epsilon=MAIN_EPSILON)
main_corrs = compute_pearson(main_results)
main_global, main_per_env = extract_metrics(main_corrs)

print(f"Bayesian Model (τ_prior={MAIN_TAU_PRIOR}, τ_softmax={MAIN_TAU_SOFTMAX}, ε={MAIN_EPSILON})")
print(f"  combo_r = {main_global['combo_r']:.2f}")
for env_id in sorted(main_per_env):
    m = main_per_env[env_id]
    print(f"    {env_id}: combo_r={m['combo_r']:.2f}")


Bayesian Model (τ_prior=5.6, τ_softmax=0.1, ε=0.2)
  combo_r = 0.40
    114_222_222_MFF: combo_r=0.63
    222_222_222_FFF: combo_r=0.41
    411_141_114_FFM: combo_r=0.54
    411_141_114_FTF: combo_r=0.22
    411_141_114_FTM: combo_r=0.49
    411_222_222_FMM: combo_r=0.05


## Run All Baselines

In [6]:
all_models = [("Bayesian (finetuned)", main_global, main_per_env)]

### 1. Random

In [7]:
preds = run_baseline_predictions(records, random_predictions)
corrs = compute_pearson(preds)
g, pe = extract_metrics(corrs)
all_models.append(("Random", g, pe))
print(f"Random: combo_r={g['combo_r']:.2f}")

Random: combo_r=0.11


### 2. Random Walk (fitting ε to maximize combo_r)

In [8]:
eps_values = np.concatenate([np.linspace(0.001, 0.1, 20), np.linspace(0.1, 1.0, 30)])
best_eps, best_combo_r = 0.5, -np.inf
rw_sweep = {}

for eps in eps_values:
    preds = run_baseline_predictions(records, random_walk_predictions, eps=eps)
    corrs = compute_pearson(preds)
    g, pe = extract_metrics(corrs)
    rw_sweep[float(eps)] = (g, pe)
    if not np.isnan(g["combo_r"]) and g["combo_r"] > best_combo_r:
        best_eps = float(eps)
        best_combo_r = g["combo_r"]

rw_global, rw_per_env = rw_sweep[best_eps]
all_models.append((f"Random Walk (ε={best_eps:.2f})", rw_global, rw_per_env))
print(f"Best ε = {best_eps:.2f}")
print(f"Random Walk: combo_r={rw_global['combo_r']:.2f}")

Best ε = 0.38
Random Walk: combo_r=0.46


### 3. Optimal

In [9]:
preds = run_baseline_predictions(records, optimal_predictions)
corrs = compute_pearson(preds)
g, pe = extract_metrics(corrs)
all_models.append(("Optimal", g, pe))
print(f"Optimal: combo_r={g['combo_r']:.2f}")

Optimal: combo_r=0.33


### 4. Random Among Top-k (fitting k to maximize combo_r)

In [10]:
best_k, best_combo_r = 1, -np.inf
topk_sweep = {}

for k in range(1, 28):
    preds = run_baseline_predictions(records, topk_predictions, k=k)
    corrs = compute_pearson(preds)
    g, pe = extract_metrics(corrs)
    topk_sweep[k] = (g, pe)
    if not np.isnan(g["combo_r"]) and g["combo_r"] > best_combo_r:
        best_k = k
        best_combo_r = g["combo_r"]
    print(f"  k={k:2d}  combo_r={g['combo_r']:.2f}")

topk_global, topk_per_env = topk_sweep[best_k]
all_models.append((f"Random Among Top-{best_k}", topk_global, topk_per_env))
print(f"\nBest k = {best_k}")
print(f"Top-k: combo_r={topk_global['combo_r']:.2f}")

  k= 1  combo_r=0.29
  k= 2  combo_r=0.33
  k= 3  combo_r=0.31
  k= 4  combo_r=0.34
  k= 5  combo_r=0.31
  k= 6  combo_r=0.33
  k= 7  combo_r=0.32
  k= 8  combo_r=0.30
  k= 9  combo_r=0.28
  k=10  combo_r=0.27
  k=11  combo_r=0.27
  k=12  combo_r=0.26
  k=13  combo_r=0.25
  k=14  combo_r=0.23
  k=15  combo_r=0.22
  k=16  combo_r=0.19
  k=17  combo_r=0.19
  k=18  combo_r=0.18
  k=19  combo_r=0.17
  k=20  combo_r=0.18
  k=21  combo_r=0.19
  k=22  combo_r=0.17
  k=23  combo_r=0.15
  k=24  combo_r=0.13
  k=25  combo_r=0.11
  k=26  combo_r=0.08
  k=27  combo_r=0.11

Best k = 4
Top-k: combo_r=0.34


### 5. Random-to-Optimal

In [11]:
preds = run_baseline_predictions(records, random_to_optimal_predictions)
corrs = compute_pearson(preds)
g, pe = extract_metrics(corrs)
all_models.append(("Random-to-Optimal", g, pe))
print(f"Random-to-Optimal: combo_r={g['combo_r']:.2f}")

Random-to-Optimal: combo_r=0.30


### 6. Copy Others

In [12]:
preds = run_baseline_predictions(records, copy_others_predictions)
corrs = compute_pearson(preds)
g, pe = extract_metrics(corrs)
all_models.append(("Copy Others", g, pe))
print(f"Copy Others: combo_r={g['combo_r']:.2f}")

Copy Others: combo_r=0.09


### 7. Contradict Others

In [13]:
preds = run_baseline_predictions(records, contradict_others_predictions)
corrs = compute_pearson(preds)
g, pe = extract_metrics(corrs)
all_models.append(("Contradict Others", g, pe))
print(f"Contradict Others: combo_r={g['combo_r']:.2f}")

Contradict Others: combo_r=0.05


## Summary Table

In [14]:
# Global summary
rows = []
for name, g, pe in all_models:
    rows.append({"Model": name, "combo_r": g["combo_r"], "marg_r": g["marg_r"]})

df_summary = pd.DataFrame(rows).set_index("Model")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)
print(df_summary.to_string(float_format="{:.4f}".format))
print()

df_summary.style.format("{:.4f}").highlight_max(axis=0, color="lightgreen")

                      combo_r  marg_r
Model                                
Bayesian (finetuned)   0.4008  0.4124
Random                 0.1052     NaN
Random Walk (ε=0.38)   0.4622  0.6878
Optimal                0.3276  0.4921
Random Among Top-4     0.3399  0.5192
Random-to-Optimal      0.2972  0.4961
Copy Others            0.0854  0.6878
Contradict Others      0.0534 -0.7477



,combo_r,marg_r
Model,,
Bayesian (finetuned),0.4008,0.4124
Random,0.1052,nan
Random Walk (ε=0.38),0.4622,0.6878
Optimal,0.3276,0.4921
Random Among Top-4,0.3399,0.5192
Random-to-Optimal,0.2972,0.4961
Copy Others,0.0854,0.6878
Contradict Others,0.0534,-0.7477


In [15]:
# Legend for column abbreviations
abbrev = {
    "Bayesian (finetuned)": "Bayes",
    "Random": "Rand",
    "Optimal": "Opt",
    "Random-to-Optimal": "R→Opt",
    "Copy Others": "Copy",
    "Contradict Others": "Contra",
}

print("Column legend:")
for name, g, pe in all_models:
    col = abbrev.get(name, name[:10])
    print(f"  {col:>10} = {name}")
print()

# Per-environment combo_r
all_envs = sorted(set(e for _, _, pe in all_models for e in pe))
rows = []
for env_id in all_envs:
    row = {"Env": env_id}
    for name, g, pe in all_models:
        col = abbrev.get(name, name[:10])
        row[col] = pe.get(env_id, {}).get("combo_r", float("nan"))
    rows.append(row)

global_row = {"Env": "GLOBAL"}
for name, g, pe in all_models:
    col = abbrev.get(name, name[:10])
    global_row[col] = g["combo_r"]
rows.append(global_row)

df_combo = pd.DataFrame(rows).set_index("Env")
print("Per-environment combo_r:")
print(df_combo.to_string(float_format="{:.2f}".format))
print()

df_combo.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen")

Column legend:
       Bayes = Bayesian (finetuned)
        Rand = Random
  Random Wal = Random Walk (ε=0.38)
         Opt = Optimal
  Random Amo = Random Among Top-4
       R→Opt = Random-to-Optimal
        Copy = Copy Others
      Contra = Contradict Others

Per-environment combo_r:
                 Bayes  Rand  Random Wal   Opt  Random Amo  R→Opt  Copy  Contra
Env                                                                            
114_222_222_MFF   0.63 -0.11        0.24  0.65        0.54   0.76 -0.03   -0.05
222_222_222_FFF   0.41  0.16        0.54  0.24        0.60   0.10  0.40   -0.30
411_141_114_FFM   0.54   NaN        0.65  0.46        0.21   0.41  0.10    0.29
411_141_114_FTF   0.22   NaN        0.64  0.16        0.27   0.14 -0.07   -0.03
411_141_114_FTM   0.49   NaN        0.63  0.41        0.51   0.22  0.06    0.21
411_222_222_FMM   0.05  0.12        0.19 -0.01        0.04  -0.04  0.16   -0.08
GLOBAL            0.40  0.11        0.46  0.33        0.34   0.30  0.09    

,Bayes,Rand,Random Wal,Opt,Random Amo,R→Opt,Copy,Contra
Env,,,,,,,,
114_222_222_MFF,0.63,-0.11,0.24,0.65,0.54,0.76,-0.03,-0.05
222_222_222_FFF,0.41,0.16,0.54,0.24,0.60,0.10,0.40,-0.30
411_141_114_FFM,0.54,nan,0.65,0.46,0.21,0.41,0.10,0.29
411_141_114_FTF,0.22,nan,0.64,0.16,0.27,0.14,-0.07,-0.03
411_141_114_FTM,0.49,nan,0.63,0.41,0.51,0.22,0.06,0.21
411_222_222_FMM,0.05,0.12,0.19,-0.01,0.04,-0.04,0.16,-0.08
GLOBAL,0.40,0.11,0.46,0.33,0.34,0.30,0.09,0.05


In [16]:
# Per-environment marg_r
rows = []
for env_id in all_envs:
    row = {"Env": env_id}
    for name, g, pe in all_models:
        col = abbrev.get(name, name[:10])
        row[col] = pe.get(env_id, {}).get("marg_r", float("nan"))
    rows.append(row)

global_row = {"Env": "GLOBAL"}
for name, g, pe in all_models:
    col = abbrev.get(name, name[:10])
    global_row[col] = g["marg_r"]
rows.append(global_row)

df_marg = pd.DataFrame(rows).set_index("Env")
print("Per-environment marg_r:")
print(df_marg.to_string(float_format="{:.2f}".format))
print()

df_marg.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen")

Per-environment marg_r:
                 Bayes  Rand  Random Wal  Opt  Random Amo  R→Opt  Copy  Contra
Env                                                                           
114_222_222_MFF   0.79   NaN        0.67 0.92        0.78   0.95  0.67   -0.76
222_222_222_FFF   0.73   NaN        0.74 0.74        0.91   0.53  0.74   -0.74
411_141_114_FFM   0.74   NaN        0.83 0.72        0.39   0.80  0.83   -0.83
411_141_114_FTF   0.45   NaN        0.83 0.68        0.82   0.85  0.83   -0.87
411_141_114_FTM   0.32   NaN        0.76  NaN        0.45  -0.16  0.76   -0.71
411_222_222_FMM  -0.35   NaN        0.43 0.03        0.03  -0.01  0.43   -0.62
GLOBAL            0.41   NaN        0.69 0.49        0.52   0.50  0.69   -0.75



,Bayes,Rand,Random Wal,Opt,Random Amo,R→Opt,Copy,Contra
Env,,,,,,,,
114_222_222_MFF,0.79,nan,0.67,0.92,0.78,0.95,0.67,-0.76
222_222_222_FFF,0.73,nan,0.74,0.74,0.91,0.53,0.74,-0.74
411_141_114_FFM,0.74,nan,0.83,0.72,0.39,0.80,0.83,-0.83
411_141_114_FTF,0.45,nan,0.83,0.68,0.82,0.85,0.83,-0.87
411_141_114_FTM,0.32,nan,0.76,nan,0.45,-0.16,0.76,-0.71
411_222_222_FMM,-0.35,nan,0.43,0.03,0.03,-0.01,0.43,-0.62
GLOBAL,0.41,nan,0.69,0.49,0.52,0.50,0.69,-0.75
